In [12]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [13]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456,0.406], std=[0.229,0.224,0.225])
])

thu_muc_chinh_train = r'D:\Deep learning Thuc hanh\Bai Thuc Hanh 2\VinaFood21\train'
dataset_train = datasets.ImageFolder(root=thu_muc_chinh_train, transform =transform)
dataLoader_train = DataLoader(dataset_train, batch_size =64, shuffle =True,num_workers=2)

thu_muc_chinh_test = r'D:\Deep learning Thuc hanh\Bai Thuc Hanh 2\VinaFood21\test'
dataset_test = datasets.ImageFolder(root=thu_muc_chinh_test,transform=transform)
dataLoader_test = DataLoader(dataset_test, batch_size =64, shuffle = False,num_workers=2)

### Bài 2: Xây dựng mô hình GoogLeNet. Huấn luyện và đánh giá mô hình GoogLeNet trên 4 độ đo precision, recall và F1 (sử dụng Adam làm optimizer). Lưu ý lớp Convolution đầu tiên có padding là 3, các lớp Max Pooling đều bật chế độ ceil_mode (`ceil_mode=True`).

In [14]:
class InceptionBlock(nn.Module):
    def __init__(self, in_ch, f1, f3r, f3, f5r, f5, fp):
        super().__init__()

        self.b1 = nn.Sequential(
            nn.Conv2d(in_ch, f1, kernel_size=1),
            nn.BatchNorm2d(f1),
            nn.ReLU(inplace=True)
        )

        self.b2 = nn.Sequential(
            nn.Conv2d(in_ch, f3r, kernel_size=1),
            nn.BatchNorm2d(f3r),
            nn.ReLU(inplace=True),
            nn.Conv2d(f3r, f3, kernel_size=3, padding=1),
            nn.BatchNorm2d(f3),
            nn.ReLU(inplace=True)
        )

        self.b3 = nn.Sequential(
            nn.Conv2d(in_ch, f5r, kernel_size=1),
            nn.BatchNorm2d(f5r),
            nn.ReLU(inplace=True),
            nn.Conv2d(f5r, f5, kernel_size=5, padding=2),
            nn.BatchNorm2d(f5),
            nn.ReLU(inplace=True)
        )
        
        self.b4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1, ceil_mode=True),
            nn.Conv2d(in_ch, fp, kernel_size=1),
            nn.BatchNorm2d(fp),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)


In [15]:
class GGnet(nn.Module):
    def __init__(self,num_classes=21):
        super().__init__()

        self.conv1 = nn.Sequential(nn.Conv2d(in_channels = 3, out_channels = 64, kernel_size = 7, stride = 2, padding =3),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace =True)
        )
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2, ceil_mode=True)

        self.conv2 = nn.Sequential(nn.Conv2d(64, 192, kernel_size=3, padding=1),
        nn.BatchNorm2d(192),
        nn.ReLU(inplace=True)
        )
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride=2, ceil_mode=True)

        self.a3 = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
        self.b3 = InceptionBlock(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride=2, ceil_mode=True)

        self.a4 = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
        self.b4 = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
        self.c4 = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
        self.d4 = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
        self.e4 = InceptionBlock(528, 256, 160, 320, 32, 128, 128)
        self.maxpool4 = nn.MaxPool2d(kernel_size=3, stride=2, ceil_mode=True)

        self.a5 = InceptionBlock(832, 256, 160, 320, 32, 128, 128) 
        self.b5 = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(1024, num_classes)
    def forward(self, x):
        x = self.maxpool1(self.conv1(x))
        x = self.maxpool2(self.conv2(x))
        x = self.a3(x)
        x = self.b3(x)
        x = self.maxpool3(x)
        x = self.a4(x)
        x = self.b4(x)
        x = self.c4(x)
        x = self.d4(x)
        x = self.e4(x)
        x = self.maxpool4(x)
        x = self.a5(x)
        x = self.b5(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GGnet(num_classes=21).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in dataLoader_train:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/total:.4f} - Acc: {correct/total:.4f}")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in dataLoader_test:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

print("Accuracy       :", accuracy_score(y_true, y_pred))
print("Precision (macro):", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Recall (macro)   :", recall_score(y_true, y_pred, average="macro", zero_division=0))
print("F1-score (macro) :", f1_score(y_true, y_pred, average="macro", zero_division=0))

del model, optimizer
torch.cuda.empty_cache()

Epoch [1/5] - Loss: 2.5475 - Acc: 0.2162
Epoch [2/5] - Loss: 2.2343 - Acc: 0.3033
Epoch [3/5] - Loss: 1.9990 - Acc: 0.3755
Epoch [4/5] - Loss: 1.8266 - Acc: 0.4195
Epoch [5/5] - Loss: 1.6840 - Acc: 0.4638
Accuracy       : 0.3392696797366058
Precision (macro): 0.3979160723909982
Recall (macro)   : 0.34614136962641673
F1-score (macro) : 0.29768276065307825


### Bài 3: Xây dựng mô hình ResNet-18, đánh giá mô hình ResNet-18 trên bộ dữ liệu VinaFood21 sử dụng các độ đo precision, recall, và F1 (Sử dụng Adam làm optimizer). Lưu ý, giữa các block Residual-Connection có một lớp Max Pooling (kernel = 3, stride = 2, paddding = 0)

In [17]:
class BasicBlock(nn.Module):
    def __init__(self,in_ch,out_ch,stride=1):
        super().__init__()

        self.conv1= nn.Conv2d(in_ch, out_ch, kernel_size=3, stride= stride,padding =1,bias = False)
        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = nn.Conv2d(out_ch,out_ch,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(out_ch)

        self.shortcut=nn.Identity()
        if stride!=1 or in_ch!=out_ch:
            self.shortcut=nn.Sequential(nn.Conv2d(in_ch,out_ch,kernel_size=1,stride=stride,bias=False),
            nn.BatchNorm2d(out_ch)
            )
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out=self.bn2(self.conv2(out))
        out=out+self.shortcut(x)
        return torch.relu(out)

In [18]:
class ResNet18(nn.Module):
    def __init__(self,num_classes=21):
        super().__init__()

        self.stem=nn.Sequential(
            nn.Conv2d(3,64,kernel_size=7,stride=2,padding=3,bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.maxpool1 = nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

        self.layer1= nn.Sequential(
            BasicBlock(in_ch=64,out_ch=64,stride=1),
            BasicBlock(in_ch=64,out_ch=64,stride=1)
        )

        self.layer2=nn.Sequential(
            BasicBlock(in_ch=64,out_ch=128,stride=2),
            BasicBlock(in_ch=128,out_ch=128,stride=1)
        )

        self.layer3=nn.Sequential(
           BasicBlock(in_ch=128,out_ch=256,stride=2),
            BasicBlock(in_ch=256,out_ch=256,stride=1)
        )

        self.layer4=nn.Sequential(
           BasicBlock(in_ch=256,out_ch=512,stride=2),
            BasicBlock(in_ch=512,out_ch=512,stride=1)
        )

        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.fc=nn.Linear(512,num_classes)

    def forward(self, x):
        x=self.stem(x)
        x=self.maxpool1(x)

        x=self.layer1(x)
        x=self.layer2(x)
        x=self.layer3(x)
        x=self.layer4(x)

        x=self.avgpool(x)
        x=torch.flatten(x,1)

        x=self.fc(x)
        return x

In [19]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18(num_classes=21).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=1e-3)
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss,correct,total=0.0,0,0
    for images,labels in dataLoader_train:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/total:.4f} - Acc: {correct/total:.4f}")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in dataLoader_test:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

print("Accuracy       :", accuracy_score(y_true, y_pred))
print("Precision (macro):", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Recall (macro)   :", recall_score(y_true, y_pred, average="macro", zero_division=0))
print("F1-score (macro) :", f1_score(y_true, y_pred, average="macro", zero_division=0))

del model, optimizer
torch.cuda.empty_cache()

Epoch [1/5] - Loss: 2.5866 - Acc: 0.2202
Epoch [2/5] - Loss: 2.2729 - Acc: 0.2975
Epoch [3/5] - Loss: 2.1226 - Acc: 0.3443
Epoch [4/5] - Loss: 1.9616 - Acc: 0.3883
Epoch [5/5] - Loss: 1.8481 - Acc: 0.4227
Accuracy       : 0.3783298413648608
Precision (macro): 0.42440507689246704
Recall (macro)   : 0.3514291094418554
F1-score (macro) : 0.33921402094533376


### Bài 4: Sử dụng pretrained ResNet50 từ HuggingFace để fine-tune trên bộ dữ liệu VinaFood21.

In [20]:
from transformers import ResNetForImageClassification

class PretrainedResnet(nn.Module):
    def __init__(self,num_classes=21,freeze_backbones=False):
        super().__init__()

        base = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
        self.resnet = base.resnet

        self.classifier = nn.Linear(2048,num_classes)

        if freeze_backbones:
            for param in self.resnet.parameters():
                param.requires_grad=False
    def forward(self,images):
        features=self.resnet(images).pooler_output
        features=torch.flatten(features,1)
        return self.classifier(features)

In [21]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model =PretrainedResnet(num_classes=21,freeze_backbones=True).to(device)
criterion = nn.CrossEntropyLoss()

optimizer=optim.Adam(filter(lambda p: p.requires_grad,model.parameters()),lr=1e-4)
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss,correct,total=0.0,0,0
    for images,labels in dataLoader_train:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/total:.4f} - Acc: {correct/total:.4f}")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in dataLoader_test:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

print("Accuracy       :", accuracy_score(y_true, y_pred))
print("Precision (macro):", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Recall (macro)   :", recall_score(y_true, y_pred, average="macro", zero_division=0))
print("F1-score (macro) :", f1_score(y_true, y_pred, average="macro", zero_division=0))

del model, optimizer
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 3799.18it/s]


Epoch [1/5] - Loss: 2.9325 - Acc: 0.0962
Epoch [2/5] - Loss: 2.8010 - Acc: 0.1634
Epoch [3/5] - Loss: 2.6916 - Acc: 0.2510
Epoch [4/5] - Loss: 2.5899 - Acc: 0.3337
Epoch [5/5] - Loss: 2.4975 - Acc: 0.3877
Accuracy       : 0.34944627357078717
Precision (macro): 0.5679455753136908
Recall (macro)   : 0.27920164782583917
F1-score (macro) : 0.276087910712692
